# Load Libraries

In [3]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline


import pandas_profiling

ModuleNotFoundError: No module named 'seaborn'

# Import Cleaned Data from Task 1

In [4]:
df=pd.read_csv('credit_clean.csv', header=0)
print(df.head())


   ID  LIMIT_BAL     SEX   EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  \
0   1      20000  female  university         1   24      2      2     -1   
1   2     120000  female  university         2   26     -1      2      0   
2   3      90000  female  university         2   34      0      0      0   
3   4      50000  female  university         1   37      0      0      0   
4   5      50000    male  university         1   57     -1      0     -1   

   PAY_4  ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0     -1  ...          0          0          0         0       689         0   
1      0  ...       3272       3455       3261         0      1000      1000   
2      0  ...      14331      14948      15549      1518      1500      1000   
3      0  ...      28314      28959      29547      2000      2019      1200   
4      0  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6      DEFAULT  
0         0    

# Define Bins/Reduce or Improve Variables
* Added Variable
 * At one point I thought a ratio of a customers bill to payment would be helpful (still might be), which is the inception of the percent_paid variable
* Bins
 * Limit_bal, age and precent_paid
* Removed Variables
 * BILL_AMT, PAY, and PAY_AMT were reduced to a single variable each


In [7]:
dfbill=df.loc[:,'BILL_AMT1':'BILL_AMT6']
dfpay=df.loc[:,'PAY_AMT1':'PAY_AMT6']
dfpaytype=df.loc[:,'PAY_0':'PAY_6']
df['BILL_AVG']=dfbill.mean(axis=1)
df['PAY_AVG']=dfpay.mean(axis=1)
df['PAY_TYPE_AVG']=dfpaytype.mean(axis=1)
df['LIMIT_BAL_BIN']=pd.cut(x = df['LIMIT_BAL'],
                        bins = [0, 100000,200000,300000,400000,500000,600000,700000,800000,900000,1000000], 
                        labels=False,
                        precision=0)
df['AGE_BIN']=pd.cut(x = df['AGE'],
                        bins = [0, 10, 20, 30, 40, 50, 60, 70, 80], 
                        labels=False,
                        precision=0)
df['BILL_AVG_BIN']=pd.cut(x = df['BILL_AVG'],
                        bins = [-1000000,0, 100000,200000,300000,400000,500000,600000,700000,800000,900000], 
                        labels=False,
                        precision=0)
df['PAY_AVG_BIN']=pd.cut(x = df['PAY_AVG'],
                        bins = [-1, 100000,200000,300000,400000,500000,600000,700000], 
                        labels=False,
                        precision=0)
df['PAY_TYPE_AVG_BIN']=pd.cut(x = df['PAY_TYPE_AVG'],
                        bins = [-2.1, -1,0,1,2,3,4,5,6], 
                        labels=False,
                        precision=0)


# Sanity Check

In [8]:
#df.dropna()
df.describe()


,ID,LIMIT_BAL,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,...,PAY_AMT5,PAY_AMT6,BILL_AVG,PAY_AVG,PAY_TYPE_AVG,LIMIT_BAL_BIN,AGE_BIN,BILL_AVG_BIN,PAY_AVG_BIN,PAY_TYPE_AVG_BIN
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,-0.291100,...,4799.387633,5215.502567,44976.945200,5275.232094,-0.182439,1.126700,2.99390,1.148900,0.001067,1.121033
std,8660.398374,129747.661567,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,1.133187,1.149988,...,15278.305679,17777.465775,63260.721860,10137.946323,0.982176,1.248772,0.95845,0.574056,0.050322,0.943617
min,1.000000,10000.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,0.000000,0.000000,-56043.166667,0.000000,-2.000000,0.000000,2.00000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,252.500000,117.750000,4781.333333,1113.291667,-0.833333,0.000000,2.00000,1.000000,0.000000,1.000000
50%,15000.500000,140000.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1500.000000,1500.000000,21051.833333,2397.166667,0.000000,1.000000,3.00000,1.000000,0.000000,1.000000
75%,22500.250000,240000.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4031.500000,4000.000000,57104.416667,5583.916667,0.000000,2.000000,4.00000,1.000000,0.000000,1.000000
max,30000.000000,1000000.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,8.000000,...,426529.000000,528666.000000,877313.833333,627344.333333,6.000000,9.000000,7.00000,9.000000,6.000000,7.000000


# Recode Catagoric Variables to Match Data Dictionary


In [9]:
df['SEX']=df['SEX'].replace({'male':1,'female':2,'other':0})
df['EDUCATION']=df['EDUCATION'].replace({'university':2, 'graduate school':1, 'high school':3, 'other':4})
df['DEFAULT']=df['DEFAULT'].replace({'default':1, 'not default':0})
dfr=df.convert_dtypes()
dfr.info()
df.to_csv('credit_cleaned_transformed.csv',index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                30000 non-null  Int64  
 1   LIMIT_BAL         30000 non-null  Int64  
 2   SEX               30000 non-null  Int64  
 3   EDUCATION         30000 non-null  Int64  
 4   MARRIAGE          30000 non-null  Int64  
 5   AGE               30000 non-null  Int64  
 6   PAY_0             30000 non-null  Int64  
 7   PAY_2             30000 non-null  Int64  
 8   PAY_3             30000 non-null  Int64  
 9   PAY_4             30000 non-null  Int64  
 10  PAY_5             30000 non-null  Int64  
 11  PAY_6             30000 non-null  Int64  
 12  BILL_AMT1         30000 non-null  Int64  
 13  BILL_AMT2         30000 non-null  Int64  
 14  BILL_AMT3         30000 non-null  Int64  
 15  BILL_AMT4         30000 non-null  Int64  
 16  BILL_AMT5         30000 non-null  Int64 

# Define Default Percentage Per Demographic and Graph

# Scratch

In [238]:
#df_dp=df[['SEX','EDUCATION','MARRIAGE','AGE_BIN','PAY_0','PERCENT_PAID_BIN','LIMIT_BAL_BIN','DEFAULT']]
#df_dp=df_dp.groupby(['SEX','EDUCATION','MARRIAGE','AGE_BIN','PAY_0','PERCENT_PAID_BIN','LIMIT_BAL_BIN']).agg({'DEFAULT': ['sum','count']})
#df_dp.columns = ['DEFAULT_SUM','DEFAULT_CT']
#df_dp=df_dp.reset_index()
#df_dp.head()
#dep_var='DEFAULT_PCT'
#for col in df_dp.columns:
#    if col!=dep_var:
#        df_dp2=df_dp[[col,dep_var]]
#        ax=df_dp2.boxplot(by=col,figsize=(5,5))
#        ax.set_xlabel(col)
#        ax.set_ylabel(dep_var)

In [ ]:
#df=pd.get_dummies(df)
#df.head()
#df=df.drop(columns=["DEFAULT_not default"])
#df=df.rename(columns={"DEFAULT_default": "DEFAULT",'EDUCATION_graduate school':'EDUCATION_graduate_school', 'EDUCATION_high school': 'EDUCATION_high school'})
